# 🎯 Optimization & Gradient Descent

Welcome to the engine room of machine learning! Every ML model trains by solving an optimization problem.

## Why Optimization?

- **Training = Optimization**: Find parameters that minimize loss
- **Gradient descent**: The workhorse algorithm
- **Variants**: SGD, Adam, RMSprop - understand the differences
- **Challenges**: Local minima, saddle points, learning rates

## What You'll Learn
1. Optimization problem formulation
2. Gradient descent algorithm
3. Learning rate effects
4. Momentum and adaptive methods
5. Stochastic gradient descent
6. Advanced optimizers (Adam, RMSprop)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import seaborn as sns
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)
%matplotlib inline

## 1. The Optimization Problem

**Goal**: Find parameters $\theta$ that minimize a loss function $L(\theta)$

$$\theta^* = \arg\min_{\theta} L(\theta)$$

**Examples**:
- Linear regression: Minimize mean squared error
- Logistic regression: Minimize cross-entropy
- Neural networks: Minimize task-specific loss

In [ ]:
# Simple 2D loss landscape
def loss_function(x, y):
    """Bowl-shaped loss function"""
    return (x - 2)**2 + (y - 1)**2

def gradient(x, y):
    """Gradient of loss function"""
    return np.array([2*(x - 2), 2*(y - 1)])

# Create meshgrid
x = np.linspace(-1, 5, 100)
y = np.linspace(-2, 4, 100)
X, Y = np.meshgrid(x, y)
Z = loss_function(X, Y)

# Visualize
fig = plt.figure(figsize=(18, 7))

# 3D surface
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8, edgecolor='none')
ax1.scatter([2], [1], [0], color='red', s=200, marker='*', 
            label='Global Minimum', zorder=10)
ax1.set_xlabel('θ₁', fontsize=12)
ax1.set_ylabel('θ₂', fontsize=12)
ax1.set_zlabel('Loss L(θ)', fontsize=12)
ax1.set_title('Loss Landscape (3D)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
fig.colorbar(surf, ax=ax1, shrink=0.5)

# 2D contour
ax2 = fig.add_subplot(122)
contour = ax2.contourf(X, Y, Z, levels=30, cmap='viridis', alpha=0.7)
contour_lines = ax2.contour(X, Y, Z, levels=30, colors='black', 
                             alpha=0.3, linewidths=0.5)
ax2.clabel(contour_lines, inline=True, fontsize=8)

# Plot gradient vectors
step = 10
for i in range(0, len(x), step):
    for j in range(0, len(y), step):
        grad = gradient(X[j, i], Y[j, i])
        grad_norm = np.linalg.norm(grad)
        if grad_norm > 0:
            grad_unit = -grad / grad_norm * 0.3  # Negative gradient direction
            ax2.arrow(X[j, i], Y[j, i], grad_unit[0], grad_unit[1],
                     head_width=0.15, head_length=0.1, fc='red', ec='red', 
                     alpha=0.6, linewidth=1.5)

ax2.plot(2, 1, 'r*', markersize=25, label='Global Minimum', zorder=10)
ax2.set_xlabel('θ₁', fontsize=12)
ax2.set_ylabel('θ₂', fontsize=12)
ax2.set_title('Loss Contours & Gradient Field', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.set_aspect('equal')
fig.colorbar(contour, ax=ax2)

plt.suptitle('🏔️ The Optimization Landscape', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("Red arrows point toward minimum (negative gradient direction)")
print("This is the direction gradient descent follows!")

## 2. Gradient Descent Algorithm

**Iterative update rule**:

$$\theta_{t+1} = \theta_t - \alpha \nabla L(\theta_t)$$

Where:
- $\alpha$ is the learning rate
- $\nabla L(\theta_t)$ is the gradient at current parameters

In [ ]:
def gradient_descent(start, learning_rate, n_iterations):
    """Run gradient descent"""
    path = [start]
    losses = [loss_function(start[0], start[1])]
    theta = start.copy()
    
    for i in range(n_iterations):
        grad = gradient(theta[0], theta[1])
        theta = theta - learning_rate * grad
        path.append(theta.copy())
        losses.append(loss_function(theta[0], theta[1]))
    
    return np.array(path), np.array(losses)

# Run gradient descent
start_point = np.array([-0.5, 3.5])
learning_rate = 0.1
n_iter = 50

path, losses = gradient_descent(start_point, learning_rate, n_iter)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Path on contour plot
contour = ax1.contourf(X, Y, Z, levels=30, cmap='viridis', alpha=0.6)
contour_lines = ax1.contour(X, Y, Z, levels=30, colors='black', 
                             alpha=0.2, linewidths=0.5)

# Plot path
ax1.plot(path[:, 0], path[:, 1], 'r.-', linewidth=2.5, markersize=8, 
         alpha=0.8, label='Gradient Descent Path')
ax1.plot(path[0, 0], path[0, 1], 'go', markersize=15, 
         label='Start', zorder=10)
ax1.plot(2, 1, 'r*', markersize=25, label='Optimum', zorder=10)
ax1.plot(path[-1, 0], path[-1, 1], 'bs', markersize=12, 
         label=f'End (iter {n_iter})', zorder=10)

# Add step numbers
for i in [0, 10, 20, 30, 40, 50]:
    if i < len(path):
        ax1.text(path[i, 0], path[i, 1] + 0.2, str(i), 
                fontsize=10, ha='center')

ax1.set_xlabel('θ₁', fontsize=12)
ax1.set_ylabel('θ₂', fontsize=12)
ax1.set_title(f'Gradient Descent Path (α={learning_rate})', 
              fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
fig.colorbar(contour, ax=ax1)

# Loss over iterations
ax2.plot(losses, 'b-', linewidth=2.5)
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('Loss Decreasing Over Time', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

# Add annotations
ax2.annotate(f'Start: {losses[0]:.2f}', xy=(0, losses[0]), 
             xytext=(5, losses[0]*2),
             arrowprops=dict(arrowstyle='->', lw=2), fontsize=11)
ax2.annotate(f'End: {losses[-1]:.6f}', xy=(n_iter, losses[-1]), 
             xytext=(n_iter-15, losses[-1]*10),
             arrowprops=dict(arrowstyle='->', lw=2), fontsize=11)

plt.suptitle('⬇️ Gradient Descent in Action', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print(f"Starting loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.6f}")
print(f"Reduction: {(1 - losses[-1]/losses[0])*100:.2f}%")
print(f"\nFinal parameters: θ = [{path[-1, 0]:.4f}, {path[-1, 1]:.4f}]")
print(f"True optimum: θ = [2.0000, 1.0000]")

## 3. The Learning Rate Problem

**Learning rate ($\alpha$)** is crucial:
- Too small: Slow convergence
- Too large: Divergence or oscillation
- Just right: Fast and stable convergence

In [ ]:
# Try different learning rates
learning_rates = [0.01, 0.1, 0.5, 0.9]
colors = ['blue', 'green', 'orange', 'red']

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

for idx, (lr, color) in enumerate(zip(learning_rates, colors)):
    ax = axes[idx]
    
    # Run gradient descent
    path, losses = gradient_descent(start_point, lr, 100)
    
    # Plot contours
    contour = ax.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.5)
    
    # Plot path
    ax.plot(path[:, 0], path[:, 1], '.-', color=color, linewidth=2, 
            markersize=6, alpha=0.8, label=f'α={lr}')
    ax.plot(path[0, 0], path[0, 1], 'go', markersize=12, label='Start')
    ax.plot(2, 1, 'r*', markersize=20, label='Optimum')
    
    # Calculate final distance from optimum
    final_dist = np.linalg.norm(path[-1] - np.array([2, 1]))
    
    ax.set_xlabel('θ₁', fontsize=11)
    ax.set_ylabel('θ₂', fontsize=11)
    
    # Determine behavior
    if lr == 0.01:
        behavior = "Too Slow"
    elif lr == 0.1:
        behavior = "Just Right ✓"
    elif lr == 0.5:
        behavior = "Oscillating"
    else:
        behavior = "Diverging!"
    
    ax.set_title(f'α = {lr} - {behavior}\nFinal distance: {final_dist:.4f}', 
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_xlim(-1, 5)
    ax.set_ylim(-2, 4)

plt.suptitle('🎚️ Effect of Learning Rate', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("Key insights:")
print("• Small α: Many small steps (slow but safe)")
print("• Medium α: Fast convergence (ideal)")
print("• Large α: Overshooting, oscillation")
print("• Very large α: Divergence (loss increases!)")

## 4. Momentum: Adding Velocity

**Problem**: Gradient descent can be slow in ravines (flat in one direction, steep in another)

**Solution**: Add momentum!

$$v_{t+1} = \beta v_t + \nabla L(\theta_t)$$
$$\theta_{t+1} = \theta_t - \alpha v_{t+1}$$

Think of it as a ball rolling downhill - builds up velocity!

In [ ]:
def gradient_descent_momentum(start, lr, momentum, n_iter):
    """Gradient descent with momentum"""
    path = [start]
    theta = start.copy()
    velocity = np.zeros_like(start)
    
    for i in range(n_iter):
        grad = gradient(theta[0], theta[1])
        velocity = momentum * velocity + grad
        theta = theta - lr * velocity
        path.append(theta.copy())
    
    return np.array(path)

# Elongated loss function (ravine)
def ravine_loss(x, y):
    return 0.1 * (x - 2)**2 + 2 * (y - 1)**2

def ravine_gradient(x, y):
    return np.array([0.2*(x - 2), 4*(y - 1)])

# Redefine gradient function temporarily
gradient_temp = gradient
gradient = ravine_gradient

# Create meshgrid for ravine
x_ravine = np.linspace(-1, 5, 100)
y_ravine = np.linspace(-1, 3, 100)
X_r, Y_r = np.meshgrid(x_ravine, y_ravine)
Z_r = ravine_loss(X_r, Y_r)

# Compare regular GD vs momentum
start = np.array([0, 2.5])
path_regular = gradient_descent(start, 0.1, 100)[0]
path_momentum = gradient_descent_momentum(start, 0.1, 0.9, 100)

# Restore original gradient
gradient = gradient_temp

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Regular GD
contour1 = ax1.contourf(X_r, Y_r, Z_r, levels=30, cmap='viridis', alpha=0.6)
ax1.plot(path_regular[:, 0], path_regular[:, 1], 'r.-', linewidth=2, 
         markersize=5, alpha=0.7, label='Regular GD')
ax1.plot(start[0], start[1], 'go', markersize=15, label='Start')
ax1.plot(2, 1, 'r*', markersize=25, label='Optimum')
ax1.set_xlabel('θ₁', fontsize=12)
ax1.set_ylabel('θ₂', fontsize=12)
ax1.set_title('Regular Gradient Descent\n(Slow in ravines)', 
              fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
fig.colorbar(contour1, ax=ax1)

# Momentum GD
contour2 = ax2.contourf(X_r, Y_r, Z_r, levels=30, cmap='viridis', alpha=0.6)
ax2.plot(path_momentum[:, 0], path_momentum[:, 1], 'b.-', linewidth=2, 
         markersize=5, alpha=0.7, label='GD with Momentum')
ax2.plot(start[0], start[1], 'go', markersize=15, label='Start')
ax2.plot(2, 1, 'r*', markersize=25, label='Optimum')
ax2.set_xlabel('θ₁', fontsize=12)
ax2.set_ylabel('θ₂', fontsize=12)
ax2.set_title('Gradient Descent with Momentum\n(Faster convergence!)', 
              fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
fig.colorbar(contour2, ax=ax2)

plt.suptitle('🚀 Momentum Accelerates Convergence', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print(f"Regular GD steps: {len(path_regular)}")
print(f"Momentum GD steps: {len(path_momentum)}")
print("\nMomentum helps escape shallow local minima and accelerate in ravines!")

## 5. Stochastic Gradient Descent (SGD)

**Problem**: Computing gradient over entire dataset is expensive

**Solution**: Use a random mini-batch!

$$\theta_{t+1} = \theta_t - \alpha \nabla L_{\text{batch}}(\theta_t)$$

**Benefits**:
- Much faster per iteration
- Can escape shallow local minima (noise helps!)
- Enables online learning

In [ ]:
# Simulate mini-batch gradient descent
def sgd_simulation(start, lr, n_iter, batch_size=10):
    """Simulate SGD with noisy gradients"""
    path = [start]
    theta = start.copy()
    
    for i in range(n_iter):
        # True gradient
        true_grad = gradient(theta[0], theta[1])
        
        # Add noise to simulate batch gradient
        noise = np.random.normal(0, 0.5, 2)
        noisy_grad = true_grad + noise
        
        theta = theta - lr * noisy_grad
        path.append(theta.copy())
    
    return np.array(path)

# Compare full-batch GD vs SGD
gradient = gradient_temp  # Use original gradient function
start = np.array([0, 3])
path_batch = gradient_descent(start, 0.1, 100)[0]
path_sgd = sgd_simulation(start, 0.1, 100)

# Visualize
fig, ax = plt.subplots(figsize=(14, 10))

contour = ax.contourf(X, Y, Z, levels=30, cmap='viridis', alpha=0.6)
contour_lines = ax.contour(X, Y, Z, levels=30, colors='black', 
                            alpha=0.2, linewidths=0.5)

# Plot paths
ax.plot(path_batch[:, 0], path_batch[:, 1], 'b.-', linewidth=2.5, 
        markersize=6, alpha=0.8, label='Batch GD (deterministic)')
ax.plot(path_sgd[:, 0], path_sgd[:, 1], 'r.-', linewidth=1.5, 
        markersize=4, alpha=0.6, label='SGD (stochastic/noisy)')
ax.plot(start[0], start[1], 'go', markersize=15, label='Start')
ax.plot(2, 1, 'y*', markersize=25, label='Optimum')

ax.set_xlabel('θ₁', fontsize=12)
ax.set_ylabel('θ₂', fontsize=12)
ax.set_title('Batch Gradient Descent vs Stochastic Gradient Descent', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
fig.colorbar(contour, ax=ax)

plt.tight_layout()
plt.show()

print("Blue path: Smooth, deterministic")
print("Red path: Noisy but faster per iteration")
print("\nSGD noise can help escape shallow local minima!")

## 6. Modern Optimizers: Adam

**Adam** (Adaptive Moment Estimation) combines:
- Momentum (first moment)
- RMSprop (second moment - adaptive learning rates)

**Why it's popular**:
- Works well out of the box
- Adapts learning rate per parameter
- Handles sparse gradients well

In [ ]:
def adam_optimizer(start, lr, n_iter, beta1=0.9, beta2=0.999, epsilon=1e-8):
    """Adam optimizer"""
    path = [start]
    theta = start.copy()
    m = np.zeros_like(start)  # First moment
    v = np.zeros_like(start)  # Second moment
    
    for t in range(1, n_iter + 1):
        grad = gradient(theta[0], theta[1])
        
        # Update moments
        m = beta1 * m + (1 - beta1) * grad
        v = beta2 * v + (1 - beta2) * (grad ** 2)
        
        # Bias correction
        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)
        
        # Update parameters
        theta = theta - lr * m_hat / (np.sqrt(v_hat) + epsilon)
        path.append(theta.copy())
    
    return np.array(path)

# Compare optimizers
start = np.array([0, 3])
path_sgd = gradient_descent(start, 0.1, 50)[0]
path_momentum = gradient_descent_momentum(start, 0.1, 0.9, 50)
path_adam = adam_optimizer(start, 0.3, 50)

# Visualize
fig, ax = plt.subplots(figsize=(14, 10))

contour = ax.contourf(X, Y, Z, levels=30, cmap='viridis', alpha=0.5)

# Plot all paths
ax.plot(path_sgd[:, 0], path_sgd[:, 1], 'b.-', linewidth=2, 
        markersize=5, alpha=0.7, label='Vanilla GD')
ax.plot(path_momentum[:, 0], path_momentum[:, 1], 'g.-', linewidth=2, 
        markersize=5, alpha=0.7, label='GD + Momentum')
ax.plot(path_adam[:, 0], path_adam[:, 1], 'r.-', linewidth=2.5, 
        markersize=6, alpha=0.8, label='Adam')
ax.plot(start[0], start[1], 'ko', markersize=15, label='Start')
ax.plot(2, 1, 'y*', markersize=30, label='Optimum')

ax.set_xlabel('θ₁', fontsize=12)
ax.set_ylabel('θ₂', fontsize=12)
ax.set_title('Comparing Optimizers', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, loc='upper right')
fig.colorbar(contour, ax=ax)

plt.tight_layout()
plt.show()

print("Optimizer comparison:")
print(f"Vanilla GD: {len(path_sgd)} steps")
print(f"Momentum: {len(path_momentum)} steps")
print(f"Adam: {len(path_adam)} steps")
print("\nAdam often converges fastest with good default hyperparameters!")

## 🚀 Key Takeaways

1. **Optimization** = finding parameters that minimize loss
2. **Gradient descent** follows negative gradient
3. **Learning rate** critically affects convergence
4. **Momentum** accelerates convergence
5. **SGD** uses mini-batches for efficiency
6. **Adam** adapts learning rates per parameter

## Choosing an Optimizer

- **Start with Adam**: Good default choice
- **SGD + Momentum**: Often better final performance
- **Tune learning rate**: Most important hyperparameter
- **Use learning rate schedules**: Reduce LR during training

## Challenges

- **Local minima**: Can get stuck (less of a problem in high dimensions)
- **Saddle points**: Flat regions where gradient ≈ 0
- **Vanishing/exploding gradients**: In deep networks
- **Learning rate selection**: Too critical!

## Resources
- Sebastian Ruder's "An overview of gradient descent optimization algorithms"
- CS231n: Optimization notes
- "Deep Learning" book by Goodfellow et al.